# 09 — Signal Validation: IC, Decay & Value Beyond Price/Volume

Validates all Phase 8 alt-data signals against the quantitative rubric:

| Test | Question | Threshold |
|------|----------|-----------|
| IC (h=1) | Does the signal rank stocks correctly next month? | |IC mean| > 0.02, NW t > 2 |
| IC decay | How long does predictive power last? | Permits: peak h=1–3; FERC: peak h=6–12 |
| Value beyond P/V | Does it add after controlling for momentum + volume? | FDR-corrected p < 0.05 |

**Prerequisites** (run before this notebook):
```
ug signals build --start 2015-01 --end 2025-12
ug model ic-test --signal all
ug model decay   --signal all
ug model cross-section --horizon 1
```

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

from urbangrowth.config import data_path, get_pipeline

pipe     = get_pipeline()
sig_dir  = data_path(pipe['processed_data_subdirs'].get('signal_tables', 'processed/signals'))

# ── Load pre-computed results (or run on-the-fly if parquets not found) ──
def _load_or_compute(parquet_glob: str, compute_fn, **kwargs):
    matches = sorted(sig_dir.glob(parquet_glob))
    if matches:
        return pd.read_parquet(matches[-1])
    print(f'No cached results found ({parquet_glob}). Running computation...')
    return compute_fn(**kwargs)

from urbangrowth.modeling.ic_test import run_full_ic_analysis
from urbangrowth.modeling.decay_test import run_decay_analysis
from urbangrowth.modeling.cross_section import value_beyond_price_volume
from urbangrowth.modeling._data import (
    build_forward_returns, build_momentum_signal, build_volume_signal,
    list_signal_sources, load_monthly_returns, load_signal_panel,
)

ic_df    = _load_or_compute('ic_results_all_*.parquet', run_full_ic_analysis)
decay_df = _load_or_compute('decay_results_all.parquet', run_decay_analysis)

print(f'IC results   : {len(ic_df):,} rows  |  sources: {ic_df["source"].nunique() if not ic_df.empty else 0}')
print(f'Decay results: {len(decay_df):,} rows  |  features: {decay_df["feature_name"].nunique() if not decay_df.empty else 0}')

## 1. IC Heatmap — Signal × Subset (h = 1 month)

In [ ]:
if ic_df.empty:
    print('No IC results. Run: ug model ic-test --signal all')
else:
    h1 = ic_df[ic_df['horizon'] == 1].copy()
    h1['label'] = h1['source'] + '/' + h1['feature_name'].str.replace('_lag\d+$', '', regex=True)
    h1_agg = h1.groupby('label')[['ic_mean', 'nw_tstat', 'fdr_rejected']].first().reset_index()
    h1_agg = h1_agg.sort_values('ic_mean', ascending=False)

    # Heatmap: rows = feature (top 30), single column = IC mean
    top_n = min(40, len(h1_agg))
    h1_top = h1_agg.head(top_n)

    fig, axes = plt.subplots(1, 2, figsize=(16, max(6, top_n * 0.35)),
                              gridspec_kw={'width_ratios': [3, 1]})

    # Left: IC mean bar
    ax = axes[0]
    colors = ['#2166ac' if v > 0 else '#d73027' for v in h1_top['ic_mean']]
    bars = ax.barh(range(len(h1_top)), h1_top['ic_mean'], color=colors, alpha=0.8)
    ax.set_yticks(range(len(h1_top)))
    ax.set_yticklabels(h1_top['label'], fontsize=7)
    ax.axvline(0, color='black', lw=0.5)
    ax.axvline(0.02, color='green', lw=0.8, linestyle='--', alpha=0.5, label='IC=0.02 threshold')
    ax.axvline(-0.02, color='red', lw=0.8, linestyle='--', alpha=0.5)
    ax.set_xlabel('IC Mean (Spearman)')
    ax.set_title('IC Mean at h=1 (top features)', fontsize=11)
    ax.legend(fontsize=8)

    # Right: NW t-stat
    ax2 = axes[1]
    tstat_colors = ['#2166ac' if abs(t) > 2 else '#aaaaaa' for t in h1_top['nw_tstat']]
    ax2.barh(range(len(h1_top)), h1_top['nw_tstat'], color=tstat_colors, alpha=0.8)
    ax2.set_yticks(range(len(h1_top)))
    ax2.set_yticklabels([])
    ax2.axvline(2, color='green', lw=0.8, linestyle='--', alpha=0.5)
    ax2.axvline(-2, color='red', lw=0.8, linestyle='--', alpha=0.5)
    ax2.set_xlabel('NW t-statistic')
    ax2.set_title('NW t-stat (|t|>2 = blue)', fontsize=11)

    # Mark FDR-significant
    for i, (_, row) in enumerate(h1_top.iterrows()):
        if row['fdr_rejected']:
            axes[0].text(h1_top['ic_mean'].max() * 1.02, i, '★', fontsize=7, va='center', color='gold')

    fig.suptitle('IC Heatmap (h=1) — ★ = FDR significant', fontsize=12)
    plt.tight_layout()
    plt.show()

    print(f'Features with |IC| > 0.02: {(h1_agg["ic_mean"].abs() > 0.02).sum()}')
    print(f'Features with |t| > 2    : {(h1_agg["nw_tstat"].abs() > 2).sum()}')
    print(f'FDR-significant (q<0.05) : {h1_agg["fdr_rejected"].sum()}')

## 2. IC Decay Curves — Horizons 0 → 12 Months

In [ ]:
if decay_df.empty:
    print('No decay results. Run: ug model decay --signal all')
else:
    sources = decay_df['source'].unique()
    n_src   = len(sources)
    ncols   = min(3, n_src)
    nrows   = (n_src + ncols - 1) // ncols

    fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 4.5 * nrows), squeeze=False)
    palette   = plt.cm.tab20.colors

    for idx, src in enumerate(sorted(sources)):
        ax   = axes[idx // ncols][idx % ncols]
        grp  = decay_df[decay_df['source'] == src]
        feats = grp['feature_name'].unique()

        for fi, feat in enumerate(sorted(feats)[:12]):  # max 12 features per panel
            fdata = grp[grp['feature_name'] == feat].sort_values('horizon')
            if fdata.empty:
                continue
            color  = palette[fi % len(palette)]
            label  = feat.replace(f'{src}_', '').replace('_', ' ')
            ax.plot(fdata['horizon'], fdata['ic_mean'],
                    marker='o', ms=4, lw=1.5, color=color, label=label, alpha=0.85)
            # 95% CI shading
            if 'ci_lo' in fdata.columns and 'ci_hi' in fdata.columns:
                ax.fill_between(fdata['horizon'], fdata['ci_lo'], fdata['ci_hi'],
                                color=color, alpha=0.08)

        ax.axhline(0, color='black', lw=0.5, linestyle=':')
        ax.axhline(0.02, color='green', lw=0.7, linestyle='--', alpha=0.4)
        ax.axhline(-0.02, color='red', lw=0.7, linestyle='--', alpha=0.4)
        ax.set_xticks([0, 1, 2, 3, 4, 6, 9, 12])
        ax.set_xlabel('Forward return horizon (months)')
        ax.set_ylabel('IC mean')
        ax.set_title(f'{src}', fontsize=10)
        if len(feats) <= 8:
            ax.legend(fontsize=6, loc='upper right', ncol=1)

    # Hide unused axes
    for idx in range(n_src, nrows * ncols):
        axes[idx // ncols][idx % ncols].set_visible(False)

    fig.suptitle('IC Decay Curves — Spearman IC by Forward Return Horizon', fontsize=13)
    plt.tight_layout()
    plt.show()

## 3. Significance Table — After BH-FDR Correction

In [ ]:
if ic_df.empty:
    print('No IC results available.')
else:
    # Full significance table across all horizons
    sig_table = ic_df.copy()
    sig_table['|t|'] = sig_table['nw_tstat'].abs()

    # Wide pivot: rows = (source, feature_name), cols = horizon
    pivot_ic = sig_table.pivot_table(
        index=['source', 'feature_name'],
        columns='horizon',
        values='ic_mean',
    ).round(4)
    pivot_t = sig_table.pivot_table(
        index=['source', 'feature_name'],
        columns='horizon',
        values='nw_tstat',
    ).round(2)
    pivot_fdr = sig_table.pivot_table(
        index=['source', 'feature_name'],
        columns='horizon',
        values='fdr_rejected',
        aggfunc='max',
    )

    # Highlight table
    def _highlight(row):
        style = []
        for v in row:
            if pd.isna(v):
                style.append('')
            elif abs(v) > 0.04:
                style.append('background-color: #2166ac; color: white; font-weight: bold')
            elif abs(v) > 0.02:
                style.append('background-color: #6baed6')
            elif v < 0:
                style.append('color: #d73027')
            else:
                style.append('')
        return style

    print('IC Mean by (signal, horizon)  |  dark blue = |IC|>0.04, light blue = |IC|>0.02')
    if len(pivot_ic) > 0:
        display(pivot_ic.style.apply(_highlight, axis=1).format('{:.4f}', na_rep='—'))
    else:
        print('(empty)')
    print()
    print('NW t-statistic by (signal, horizon)')
    if len(pivot_t) > 0:
        display(pivot_t)

    # Count passing the bar
    n_h1_sig = sig_table[(sig_table['horizon'] == 1) & (sig_table['fdr_rejected'])].shape[0]
    n_any_sig = sig_table[sig_table['fdr_rejected']].shape[0]
    print(f'\nFDR-significant at h=1   : {n_h1_sig}')
    print(f'FDR-significant (any h)  : {n_any_sig}')

## 4. Value Beyond Price/Volume — Cross-Sectional Regression

In [ ]:
# Load pre-computed or run on-the-fly
pv_results = _load_or_compute(
    'cross_section_h1.parquet',
    lambda: (
        __import__('urbangrowth.modeling.cross_section', fromlist=['run']).run(horizon=1)
        or pd.DataFrame()
    ),
)

if isinstance(pv_results, pd.DataFrame) and not pv_results.empty:
    pv = pv_results.copy()
    pv['significant'] = pv.get('fdr_rejected', False)
    pv_h1 = pv[pv['horizon'] == 1].sort_values('nw_tstat', ascending=False)

    fig, ax = plt.subplots(figsize=(14, max(5, len(pv_h1) * 0.3)))
    y_pos   = range(len(pv_h1))
    colors  = ['#2166ac' if s else '#aaaaaa' for s in pv_h1['significant']]
    ax.barh(y_pos, pv_h1['nw_tstat'], color=colors, alpha=0.85)
    ax.axvline(2, color='green', lw=1, linestyle='--', alpha=0.6, label='t=2')
    ax.axvline(-2, color='red', lw=1, linestyle='--', alpha=0.6)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(
        [f"{r['source']}/{r['feature_name']}" for _, r in pv_h1.iterrows()],
        fontsize=7,
    )
    ax.set_xlabel('NW t-statistic of alt signal (after controlling for momentum + volume)')
    ax.set_title('Value Beyond Price/Volume — h=1  |  Blue = FDR significant', fontsize=11)
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.show()

    print('\nFDR-significant signals (incremental value beyond momentum + volume):')
    sig_pv = pv_h1[pv_h1['significant']]
    if sig_pv.empty:
        print('  None pass FDR correction at h=1.')
    else:
        print(sig_pv[['source', 'feature_name', 'mean_slope', 'nw_tstat', 'p_adjusted']].to_string())
else:
    print('No cross-section results. Run: ug model cross-section --horizon 1')

## 5. Pitch-Ready Summary

Which signals survive the full validation rubric for which ticker subsets?

In [ ]:
if ic_df.empty:
    print('No data available.')
else:
    # Rubric: IC > 0.02, |t| > 2, FDR rejected at h=1 or h=2
    h12 = ic_df[ic_df['horizon'].isin([1, 2])]
    passing = h12[
        (h12['ic_mean'].abs() > 0.02) &
        (h12['nw_tstat'].abs() > 2.0) &
        (h12['fdr_rejected'] == True)
    ].copy()

    if passing.empty:
        print('No signals pass the full rubric with current data.')
        print('Possible causes:')
        print('  • Insufficient returns history (need 3+ years)')
        print('  • Signal features not yet populated (run ug signals build)')
        print('  • Cross-section too small (<10 stocks with non-NaN signal per month)')
    else:
        # Group by source → show which families deliver
        summary = (
            passing.groupby(['source', 'horizon'])
            .agg(
                n_features=('feature_name', 'nunique'),
                best_ic=('ic_mean', 'max'),
                best_t=('nw_tstat', lambda x: x.abs().max()),
                top_feature=('feature_name', lambda x: x.iloc[x.reset_index(drop=True).index[0]]),
            )
            .reset_index()
            .sort_values(['source', 'horizon'])
        )

        print('=' * 70)
        print('SIGNALS PASSING FULL RUBRIC (IC>0.02, |t|>2, FDR q<0.05)')
        print('=' * 70)
        for _, row in summary.iterrows():
            print(f"  {row['source']:20s}  h={int(row['horizon'])}  "
                  f"{row['n_features']} features  "
                  f"best IC={row['best_ic']:.4f}  |t|={row['best_t']:.2f}")
            print(f"    Top feature: {row['top_feature']}")
        print()

        # Expected patterns check
        print('Expected pattern checks:')
        permit_peak = decay_df[(decay_df['source']=='census_bps')].groupby('horizon')['ic_mean'].mean() if not decay_df.empty else pd.Series()
        ferc_peak   = decay_df[(decay_df['source']=='ferc_queue')].groupby('horizon')['ic_mean'].mean() if not decay_df.empty else pd.Series()

        if not permit_peak.empty:
            best_lag = permit_peak.idxmax()
            expected = '✓' if best_lag in (1, 2, 3) else '✗'
            print(f'  Permit peak lag = {best_lag}m  {expected} (expected 1–3m)')
        if not ferc_peak.empty:
            best_lag = ferc_peak.idxmax()
            expected = '✓' if best_lag >= 6 else '✗'
            print(f'  FERC peak lag   = {best_lag}m  {expected} (expected 6–12m)')